In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Connect to Northwind SQLite DB
conn = sqlite3.connect('northwind.db') # Ensure northwind.db is in your directory

In [ ]:
query_top_products = """
SELECT p.ProductName, SUM(od.Quantity) AS TotalQuantitySold
FROM Products p
JOIN [Order Details] od ON p.ProductID = od.ProductID
GROUP BY p.ProductID, p.ProductName
ORDER BY TotalQuantitySold DESC
LIMIT 10;
"""
df_top_products = pd.read_sql_query(query_top_products, conn)
print(df_top_products)

# Visualization
plt.figure(figsize=(10, 5))
sns.barplot(data=df_top_products, x='TotalQuantitySold', y='ProductName', palette='Blues_r')
plt.title('Top 10 Selling Products by Quantity')
plt.xlabel('Quantity Sold')
plt.ylabel('Product Name')
plt.show()

In [ ]:
query_top_customers = """
SELECT c.CompanyName, ROUND(SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)), 2) AS TotalSpent
FROM Customers c
JOIN Orders o ON c.CustomerID = o.CustomerID
JOIN [Order Details] od ON o.OrderID = od.OrderID
GROUP BY c.CustomerID, c.CompanyName
ORDER BY TotalSpent DESC
LIMIT 10;
"""
df_top_customers = pd.read_sql_query(query_top_customers, conn)
print(df_top_customers)

In [ ]:
query_monthly = """
SELECT strftime('%Y-%m', o.OrderDate) AS YearMonth,
       ROUND(SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)), 2) AS MonthlyRevenue
FROM Orders o
JOIN [Order Details] od ON o.OrderID = od.OrderID
GROUP BY YearMonth
ORDER BY YearMonth ASC;
"""
df_monthly = pd.read_sql_query(query_monthly, conn)

plt.figure(figsize=(12, 5))
plt.plot(df_monthly['YearMonth'], df_monthly['MonthlyRevenue'], marker='o', color='b')
plt.xticks(rotation=45)
plt.title('Monthly Sales Revenue Trend')
plt.xlabel('Month')
plt.ylabel('Revenue ($)')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
query_categories = """
SELECT cat.CategoryName, ROUND(SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)), 2) AS TotalCategoryRevenue
FROM Categories cat
JOIN Products p ON cat.CategoryID = p.CategoryID
JOIN [Order Details] od ON p.ProductID = od.ProductID
GROUP BY cat.CategoryID, cat.CategoryName
ORDER BY TotalCategoryRevenue DESC;
"""
df_categories = pd.read_sql_query(query_categories, conn)

plt.figure(figsize=(8, 8))
plt.pie(df_categories['TotalCategoryRevenue'], labels=df_categories['CategoryName'], autopct='%1.1f%%', startangle=140)
plt.title('Revenue Contribution by Product Category')
plt.show()

In [ ]:
query_freq = """
SELECT c.CompanyName, COUNT(o.OrderID) AS OrderFrequency
FROM Customers c
LEFT JOIN Orders o ON c.CustomerID = o.CustomerID
GROUP BY c.CustomerID, c.CompanyName
ORDER BY OrderFrequency DESC;
"""
df_freq = pd.read_sql_query(query_freq, conn)
print(df_freq.describe()) # Exploratory summary